<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session10-discussion-1.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 10: in-class discussion problem (1 of 3)

This is the block 1 question of the lecture quiz. First guess the answer in the quiz, on your own.
Then fix and run the code below, discuss the outcome with your neighbour, and answer the quiz again.

## How many weights does a convolutional layer have?

A convolutional layer with 64 filters, each $3 \times 3$, reads an RGB image (3 channels):
`nn.Conv2d(3, 64, kernel_size=3)`. How many weights does it have on a $224 \times 224$ image, and on a
$448 \times 448$ image? (Leave out the 64 biases.)

**Fix and run.** The function in the cell below is meant to count the weights by hand, but the line
marked `FIX` counts as if every filter looked at the whole image. Run it once as it is, and compare
with PyTorch's own count on the last line. Then fix the function and run it again.

In [1]:
import torch.nn as nn

C_in, C_out, k = 3, 64, 3        # RGB in, 64 filters out, 3 x 3 kernels

def conv_weights(H, W):
    """Weights in nn.Conv2d(C_in, C_out, kernel_size=k) on an H x W image (biases left out)."""
    return H * W * C_in * C_out  # FIX: what does one filter actually look at?

for H in (224, 448):
    print(f"{H} x {H} image: {conv_weights(H, H):>12,} weights")

# Check against PyTorch's own layer
layer = nn.Conv2d(C_in, C_out, kernel_size=k, bias=False)
print(f"PyTorch's nn.Conv2d:  {layer.weight.numel():>12,} weights, shape {tuple(layer.weight.shape)}")

224 x 224 image:    9,633,792 weights
448 x 448 image:   38,535,168 weights
PyTorch's nn.Conv2d:         1,728 weights, shape (64, 3, 3, 3)


**If you finish early.**

1. The unfixed function gave the count for a *fully connected* layer with 64 output units. How many
   times more weights is that than the convolution, at $224 \times 224$ and at $448 \times 448$?
2. Is a 2D convolution translation-equivariant, like the 1D one in the main notebook? The two cells
   below shift a single bright pixel by (7, 7) and compare the outputs.

In [2]:
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

In [3]:
conv2d = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=5, padding=2, bias=False)

size = 40
image = torch.zeros(1, 1, size, size)
image[0, 0, 15, 15] = 1.0

shift = 7
shifted_image = torch.zeros(1, 1, size, size)
shifted_image[0, 0, 15 + shift, 15 + shift] = 1.0

with torch.no_grad():
    out = conv2d(image)[0, 0].numpy()
    out_shifted = conv2d(shifted_image)[0, 0].numpy()

out_rolled = np.roll(np.roll(out, shift, axis=0), shift, axis=1)
interior = (slice(10, 30), slice(10, 30))  # stay away from both edges in both dimensions
max_abs_diff = np.max(np.abs(out_rolled[interior] - out_shifted[interior]))
print(f"max |shift(conv2d(x)) - conv2d(shift(x))| over the interior region: {max_abs_diff:.2e}")
print("2D equivariant too (matches to floating-point precision away from the edges):", max_abs_diff < 1e-6)

max |shift(conv2d(x)) - conv2d(shift(x))| over the interior region: 0.00e+00
2D equivariant too (matches to floating-point precision away from the edges): True


**Discussion point.** One filter looks at a $3 \times 3$ patch across all 3 input channels:
$3 \cdot 3 \cdot 3 = 27$ weights. With 64 filters that is $27 \cdot 64 = 1{,}728$ weights, the shape
`(64, 3, 3, 3)` that PyTorch reports. The fix is

```python
return k * k * C_in * C_out
```

and the count does not depend on the image size at all: the same 1,728 weights for $224 \times 224$
and for $448 \times 448$, because the same small kernel is reused at every position (weight sharing).

The unfixed line, $H \cdot W \cdot C_\text{in} \cdot C_\text{out}$, is the count for a fully
connected layer with 64 units: 9,633,792 weights at $224 \times 224$ (5,575 times more) and 38,535,168
at $448 \times 448$, four times as many again, since doubling each side quadruples the number of
pixels. That is the parameter explosion that made dense layers unusable for images, and the shared
kernel is also what makes the convolution equivariant: shift the input, and the output shifts with
it.